# Sprint 2 – Ingeniería de Datos y Pipeline Reproducible
**Proyecto:** Forecast de Ingresos Mensuales – Olist E-Commerce  
**Target:** `monthly_revenue`  
**Tipo:** Series Temporales  
**Métricas:** RMSE, MAPE (objetivo: MAPE < 10%)  

## Flujo del Sprint 2
| Paso | Descripción |
|------|-------------|
| 1 | EDA Básico + carga de datos |
| 2 | Definición de splits temporales (Train/Val/BackTest/Live/Predict) |
| 3 | Generación de Features (40-60 features) |
| 4 | Creación de la Master Table mensual |
| 5 | Limpieza de variables (Clip, NaN, Agrupamiento) |
| 6 | Selección de Variables (Missing, PSI, Correlación, Univariante, Wrapper) |
| 7 | Hiperparametrización con Optuna |
| 8 | Aplicar Modelo Final |


In [1]:
# ── Setup: ajustar ruta al proyecto
import sys, os
PROJECT_ROOT = '..'  # Ajustar si el notebook está en otra ubicación
sys.path.insert(0, PROJECT_ROOT)
os.chdir(PROJECT_ROOT)

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import plotly.graph_objects as go
import plotly.express as px
from plotly.subplots import make_subplots

from src.utils.helpers import load_config, setup_logger
setup_logger('sprint2.log')

cfg = load_config()
print(' Setup completado')
print(f'   Proyecto: {cfg["project"]["name"]} v{cfg["project"]["version"]}')
print(f'   Target:   {cfg["project"]["target"]}')
print(f'   Horizonte: {cfg["project"]["horizon_months"]} meses')

 Setup completado
   Proyecto: olist_revenue_forecast v1.0.0
   Target:   monthly_revenue
   Horizonte: 3 meses


## Paso 1 y 2 - Carga de Datos y Splits Temporales

In [2]:
from src.data.loader import load_raw_datasets, parse_dates, data_quality_report

# ─── Ajustar DATA_PATH según tu entorno
# DATA_PATH = '/content/drive/MyDrive/MAESTRIA/Modulo13/dev/data/'  # Colab
DATA_PATH = None  # None = usa config (kaggle download o data/raw/)

datasets = load_raw_datasets(cfg, DATA_PATH)
datasets = parse_dates(datasets)
quality_report = data_quality_report(datasets)

2026-06-06 09:11:45.480 | INFO     | src.data.loader:load_raw_datasets:99 -   orders              :   99,441 filas x   8 cols
2026-06-06 09:11:46.032 | INFO     | src.data.loader:load_raw_datasets:99 -   order_items         :  112,650 filas x   7 cols
2026-06-06 09:11:46.222 | INFO     | src.data.loader:load_raw_datasets:99 -   order_payments      :  103,886 filas x   5 cols
2026-06-06 09:11:46.982 | INFO     | src.data.loader:load_raw_datasets:99 -   order_reviews       :   99,224 filas x   7 cols
2026-06-06 09:11:47.311 | INFO     | src.data.loader:load_raw_datasets:99 -   customers           :   99,441 filas x   5 cols
2026-06-06 09:11:47.442 | INFO     | src.data.loader:load_raw_datasets:99 -   products            :   32,951 filas x   9 cols
2026-06-06 09:11:47.455 | INFO     | src.data.loader:load_raw_datasets:99 -   sellers             :    3,095 filas x   4 cols
2026-06-06 09:11:47.461 | INFO     | src.data.loader:load_raw_datasets:99 -   category_trans      :       71 filas x  

## Paso 3 y 4 - Master Table y Features Mensuales

In [3]:
from src.data.master_table import build_master_table, add_transaction_features
from src.data.monthly_agg import build_monthly_table, add_time_series_features, split_data

# Construcción
df_master = build_master_table(datasets)
df_master = add_transaction_features(df_master)

# Tabla mensual con target
monthly = build_monthly_table(df_master)
monthly = add_time_series_features(monthly, cfg)

print(f'\n Master Table mensual:')
print(f'   {len(monthly)} meses  x  {monthly.shape[1]} features')
monthly[['ds', 'monthly_revenue', 'monthly_orders', 'avg_ticket']].head(10)

2026-06-06 09:12:20.785 | INFO     | src.data.master_table:build_master_table:21 - Construyendo Master Table…
2026-06-06 09:12:22.206 | INFO     | src.data.master_table:build_master_table:61 - Master Table base: 119,143 filas x 30 cols
2026-06-06 09:12:22.210 | INFO     | src.data.master_table:add_transaction_features:71 - Generando features transaccionales…
2026-06-06 09:12:22.576 | INFO     | src.data.master_table:add_transaction_features:126 - Features transaccionales añadidas: 54 cols totales
2026-06-06 09:12:22.579 | INFO     | src.data.monthly_agg:build_monthly_table:26 - Construyendo tabla mensual…
2026-06-06 09:12:22.977 | INFO     | src.data.monthly_agg:build_monthly_table:36 - Órdenes válidas (delivered + payment>0): 96,477
2026-06-06 09:12:23.344 | INFO     | src.data.monthly_agg:build_monthly_table:69 - Serie mensual: 20 meses  (Dec 2016 → Jul 2018)
2026-06-06 09:12:23.401 | INFO     | src.data.monthly_agg:add_time_series_features:119 - Features de series temporales añadida


 Master Table mensual:
   20 meses  x  44 features


,ds,monthly_revenue,monthly_orders,avg_ticket
0,2016-12-01,19.62,1,19.620000
1,2017-01-01,178282.10,750,181.735066
2,2017-02-01,327928.86,1653,166.208241
3,2017-03-01,508767.44,2546,164.224480
4,2017-04-01,457050.31,2303,168.342656
5,2017-05-01,707042.90,3546,164.161342
6,2017-06-01,590223.90,3135,158.406844
7,2017-07-01,720446.68,3872,151.005383
8,2017-08-01,850611.08,4193,166.557877
9,2017-09-01,1003326.07,4150,199.072633


In [4]:
# Splits temporales
splits = split_data(monthly, cfg)

print('\n SPLITS TEMPORALES:')
print('='*50)
for name, df in splits.items():
    if len(df) > 0:
        print(f'  {name:12s}: {len(df):>3} meses  '
              f'({df["ds"].min().strftime("%b %Y")} → '
              f'{df["ds"].max().strftime("%b %Y")})')

2026-06-06 09:12:34.499 | INFO     | src.data.monthly_agg:split_data:154 -   Split train     :  17 meses
2026-06-06 09:12:34.500 | INFO     | src.data.monthly_agg:split_data:154 -   Split val       :   3 meses
2026-06-06 09:12:34.501 | INFO     | src.data.monthly_agg:split_data:154 -   Split backtest  :   0 meses
2026-06-06 09:12:34.504 | INFO     | src.data.monthly_agg:split_data:154 -   Split live      :   0 meses
2026-06-06 09:12:34.505 | INFO     | src.data.monthly_agg:split_data:154 -   Split all       :  20 meses



 SPLITS TEMPORALES:
  train       :  17 meses  (Dec 2016 → Apr 2018)
  val         :   3 meses  (May 2018 → Jul 2018)
  all         :  20 meses  (Dec 2016 → Jul 2018)


## Paso 5 Limpieza de Variables - Clip + NaN + Agrupamiento

In [5]:
from src.features.cleaning import clean_monthly_table

print('🔧 PASO 5: Limpieza de Variables')
print('='*50)
print(f'  Clip quantiles: [{cfg["cleaning"]["clip_quantile_low"]}, {cfg["cleaning"]["clip_quantile_high"]}]')
print(f'  NaN strategy:   {cfg["cleaning"]["nan_fill_strategy"]}')
print(f'  Rare threshold: {cfg["cleaning"]["group_rare_threshold"]*100}%')

monthly_clean, cleaning_pipeline = clean_monthly_table(monthly, cfg, fit=True)

print(f'\n MT Limpiada: {monthly_clean.shape}')
print(f'   NaN restantes: {monthly_clean.isnull().sum().sum()}')

2026-06-06 09:12:47.940 | INFO     | src.features.cleaning:fit:44 - OutlierClipper ajustado en 41 columnas
2026-06-06 09:12:47.946 | DEBUG    | src.features.cleaning:transform:54 -   monthly_orders: 2 valores clipados


🔧 PASO 5: Limpieza de Variables
  Clip quantiles: [0.01, 0.99]
  NaN strategy:   median
  Rare threshold: 2.0%


2026-06-06 09:12:47.951 | DEBUG    | src.features.cleaning:transform:54 -   monthly_customers: 2 valores clipados
2026-06-06 09:12:47.958 | DEBUG    | src.features.cleaning:transform:54 -   monthly_items: 2 valores clipados
2026-06-06 09:12:47.963 | DEBUG    | src.features.cleaning:transform:54 -   avg_ticket: 2 valores clipados
2026-06-06 09:12:47.967 | DEBUG    | src.features.cleaning:transform:54 -   median_ticket: 2 valores clipados
2026-06-06 09:12:47.972 | DEBUG    | src.features.cleaning:transform:54 -   avg_review_score: 2 valores clipados
2026-06-06 09:12:47.979 | DEBUG    | src.features.cleaning:transform:54 -   avg_delivery_days: 2 valores clipados
2026-06-06 09:12:47.984 | DEBUG    | src.features.cleaning:transform:54 -   avg_delay_days: 2 valores clipados
2026-06-06 09:12:47.990 | DEBUG    | src.features.cleaning:transform:54 -   pct_late_delivery: 2 valores clipados
2026-06-06 09:12:47.995 | DEBUG    | src.features.cleaning:transform:54 -   avg_freight: 2 valores clipados


 MT Limpiada: (20, 44)
   NaN restantes: 0


## Paso 6 Selección de Variables

In [7]:
from src.features.selection import run_feature_selection, get_selection_summary

# Re-split con datos limpios
splits_clean = split_data(monthly_clean, cfg)
train = splits_clean['train']
val   = splits_clean['val']
if len(val) == 0:
    val = train.tail(3)

print(' PASO 6: Selección de Variables')
print('='*50)

selection_result = run_feature_selection(
    monthly_train=train,
    monthly_val=val,
    target_col=cfg['project']['target'],
    cfg=cfg,
)

selected_features = selection_result['selected_features']
summary = get_selection_summary(selection_result['steps_report'])
print('\n Resumen de selección:')
print(summary.to_string(index=False))
print(f'\n Features seleccionadas ({len(selected_features)}):')
for i, f in enumerate(selected_features, 1):
    print(f'  {i:2}. {f}')

2026-06-06 09:14:16.975 | INFO     | src.data.monthly_agg:split_data:154 -   Split train     :  17 meses
2026-06-06 09:14:16.976 | INFO     | src.data.monthly_agg:split_data:154 -   Split val       :   3 meses
2026-06-06 09:14:16.979 | INFO     | src.data.monthly_agg:split_data:154 -   Split backtest  :   0 meses
2026-06-06 09:14:16.981 | INFO     | src.data.monthly_agg:split_data:154 -   Split live      :   0 meses
2026-06-06 09:14:16.983 | INFO     | src.data.monthly_agg:split_data:154 -   Split all       :  20 meses
2026-06-06 09:14:16.990 | INFO     | src.features.selection:run_feature_selection:268 - 
2026-06-06 09:14:16.992 | INFO     | src.features.selection:run_feature_selection:269 - SELECCIÓN DE VARIABLES – Estado inicial: 41 features
2026-06-06 09:14:16.995 | INFO     | src.features.selection:run_feature_selection:270 - ============================================================
2026-06-06 09:14:17.010 | INFO     | src.features.selection:missing_selection:97 - Missing: 41 f

 PASO 6: Selección de Variables


2026-06-06 09:14:23.811 | INFO     | src.features.selection:wrapper_selection:214 - WRAPPER RFECV: 5 features seleccionadas
2026-06-06 09:14:23.812 | INFO     | src.features.selection:run_feature_selection:372 - 
2026-06-06 09:14:23.815 | INFO     | src.features.selection:run_feature_selection:373 - RESUMEN DE SELECCIÓN DE VARIABLES:
2026-06-06 09:14:23.817 | INFO     | src.features.selection:run_feature_selection:375 -   0_initial: 41 features
2026-06-06 09:14:23.819 | INFO     | src.features.selection:run_feature_selection:375 -   1_missing: 41 features
2026-06-06 09:14:23.822 | INFO     | src.features.selection:run_feature_selection:375 -   2_psi: 5 features
2026-06-06 09:14:23.824 | INFO     | src.features.selection:run_feature_selection:375 -   3_correlation: 5 features
2026-06-06 09:14:23.827 | INFO     | src.features.selection:run_feature_selection:375 -   4_univariate: 5 features
2026-06-06 09:14:23.830 | INFO     | src.features.selection:run_feature_selection:375 -   5_wrapper


 Resumen de selección:
         step  features_remaining  threshold
    0_initial                  41        NaN
    1_missing                  41       0.60
        2_psi                   5        NaN
3_correlation                   5       0.95
 4_univariate                   5       0.10
    5_wrapper                   5        NaN

 Features seleccionadas (5):
   1. monthly_orders
   2. month
   3. avg_delay_days
   4. month_sin
   5. quarter


## Paso 7 Hiperparametrización con Optuna

In [8]:
from src.models.trainer import tune_lightgbm, plot_optimization_history

print('⚡ PASO 7: Hiperparametrización (Optuna)')
print('='*50)

# Reducir n_trials para demo (aumentar para producción)
cfg_demo = cfg.copy()
cfg_demo['optuna']['n_trials'] = 20
cfg_demo['optuna']['timeout']  = 120

X_train = train[selected_features]
y_train = train[cfg['project']['target']]

tuning_result = tune_lightgbm(X_train, y_train, selected_features, cfg_demo)
best_params = tuning_result['best_params']

print(f'\n Mejor RMSE en CV: R$ {tuning_result["best_value"]:,.2f}')
print('\n Mejores hiperparámetros:')
for k, v in best_params.items():
    print(f'   {k}: {v}')

# Visualizar historial de optimización
plot_optimization_history(tuning_result['study'])

2026-06-06 09:15:45.222 | INFO     | src.models.trainer:tune_lightgbm:95 - Iniciando Optuna (n_trials=20, timeout=120s, metric=rmse)…


⚡ PASO 7: Hiperparametrización (Optuna)


Best trial: 6. Best value: 460626: 100%|██████████| 20/20 [00:05<00:00,  3.55it/s, 5.62/120 seconds]
2026-06-06 09:15:50.865 | INFO     | src.models.trainer:tune_lightgbm:119 - Mejor RMSE: 460626.2857
2026-06-06 09:15:50.867 | INFO     | src.models.trainer:tune_lightgbm:120 - Mejores parámetros: {'n_estimators': 638, 'learning_rate': 0.0835361075531176, 'num_leaves': 19, 'max_depth': 4, 'min_child_samples': 6, 'subsample': 0.7301321323053057, 'colsample_bytree': 0.7554709158757928, 'reg_alpha': 0.2713490317738959, 'reg_lambda': 0.8287375091519293}



 Mejor RMSE en CV: R$ 460,626.29

 Mejores hiperparámetros:
   n_estimators: 638
   learning_rate: 0.0835361075531176
   num_leaves: 19
   max_depth: 4
   min_child_samples: 6
   subsample: 0.7301321323053057
   colsample_bytree: 0.7554709158757928
   reg_alpha: 0.2713490317738959
   reg_lambda: 0.8287375091519293


## Paso 8 Modelo Final – Evaluación y Comparación

In [9]:
from src.models.trainer import train_final_model
from src.models.baseline import evaluate_baselines
from src.models.forecaster import ProphetForecaster
from src.evaluation.metrics import (
    compute_technical_metrics, compare_models, format_business_report
)

# ── Baseline
print(' BASELINE MODELS:')
backtest = splits_clean['backtest']
test_data = backtest if len(backtest) > 0 else splits_clean['val']
baseline_results = evaluate_baselines(train, test_data, n_test=min(3, len(test_data)))
print(baseline_results.to_string(index=False))

print('\n' + '='*60)

# ── Modelo Final LightGBM
print(' MODELO FINAL (LightGBM):')
all_data = splits_clean['all']
target   = cfg['project']['target']
horizon  = cfg['project']['horizon_months']

forecaster_lgbm = train_final_model(
    all_data[selected_features],
    all_data[target],
    best_params,
    selected_features,
    horizon=horizon
)

# Evaluar en backtest
if len(test_data) > 0:
    X_test = test_data[selected_features]
    y_test = test_data[target].values[:horizon]
    y_pred = forecaster_lgbm.predict(X_test)[:len(y_test)]
    
    lgbm_metrics = compute_technical_metrics(y_test, y_pred, 'LightGBM_final')
    
    print(f'  RMSE : R$ {lgbm_metrics["rmse"]:>12,.2f}')
    print(f'  MAE  : R$ {lgbm_metrics["mae"]:>12,.2f}')
    print(f'  MAPE :    {lgbm_metrics["mape"]:>10.2f}%')
    print(f'  sMAPE:    {lgbm_metrics["smape"]:>10.2f}%')
    
    if lgbm_metrics['mape'] < 10.0:
        print('\n OBJETIVO ALCANZADO: MAPE < 10%')
    else:
        print(f'\n  MAPE = {lgbm_metrics["mape"]:.2f}% > 10% – considerar más features o datos')

2026-06-06 09:15:59.420 | INFO     | src.evaluation.metrics:compute_technical_metrics:51 - [Naive] RMSE=139,874.73  MAE=117,930.31  MAPE=9.05%  sMAPE=8.51%
2026-06-06 09:15:59.422 | INFO     | src.evaluation.metrics:compute_technical_metrics:51 - [MovingAverage_3M] RMSE=96,803.98  MAE=95,645.99  MAPE=7.12%  sMAPE=6.96%
2026-06-06 09:15:59.425 | INFO     | src.evaluation.metrics:compute_technical_metrics:51 - [SeasonalNaive] RMSE=691,070.88  MAE=686,897.28  MAPE=50.47%  sMAPE=67.69%
2026-06-06 09:15:59.443 | INFO     | src.evaluation.metrics:compute_technical_metrics:51 - [LinearTrend] RMSE=423,893.54  MAE=395,532.86  MAPE=29.87%  sMAPE=25.45%
2026-06-06 09:15:59.452 | INFO     | src.models.baseline:evaluate_baselines:114 - 
Mejor baseline: MovingAverage_3M (MAPE=7.12%)
2026-06-06 09:15:59.457 | INFO     | src.models.baseline:evaluate_baselines:115 - Objetivo final: MAPE < 10%


 BASELINE MODELS:
           model          rmse           mae      mape     smape
MovingAverage_3M  96803.975771  95645.988889  7.120480  6.961316
           Naive 139874.726733 117930.313333  9.049090  8.506213
     LinearTrend 423893.542589 395532.863578 29.866995 25.451590
   SeasonalNaive 691070.878606 686897.276667 50.471682 67.691024

 MODELO FINAL (LightGBM):


2026-06-06 09:16:00.114 | INFO     | src.models.forecaster:fit:82 - LightGBM entrenado (3 horizontes)
2026-06-06 09:16:00.118 | INFO     | src.models.trainer:train_final_model:158 - Modelo final entrenado con mejores hiperparámetros
2026-06-06 09:16:00.132 | INFO     | src.evaluation.metrics:compute_technical_metrics:51 - [LightGBM_final] RMSE=112,109.04  MAE=102,186.09  MAPE=7.71%  sMAPE=7.96%


  RMSE : R$   112,109.04
  MAE  : R$   102,186.09
  MAPE :          7.71%
  sMAPE:          7.96%

 OBJETIVO ALCANZADO: MAPE < 10%


In [10]:
# ── Importancia de features
fi = forecaster_lgbm.feature_importance()

fig = px.bar(
    fi.head(20), x='importance', y='feature',
    orientation='h', color='importance',
    color_continuous_scale='Blues',
    title='Top 20 Features – Importancia LightGBM'
)
fig.update_layout(height=500, yaxis=dict(autorange='reversed'))
fig.show()

In [11]:
# ── Forecast final: próximos 3 meses
print(' FORECAST – Próximos 3 meses:')
print('='*50)

X_last = monthly_clean[selected_features].tail(1)
y_forecast = forecaster_lgbm.predict(X_last)

last_date = monthly_clean['ds'].max()
for i, val in enumerate(y_forecast, 1):
    future_date = last_date + pd.DateOffset(months=i)
    ci_low  = val * 0.85
    ci_high = val * 1.15
    print(f'  {future_date.strftime("%b %Y")}: '
          f'R$ {val:>12,.2f}  '
          f'[R$ {ci_low:>12,.2f} – R$ {ci_high:>12,.2f}]')

 FORECAST – Próximos 3 meses:
  Aug 2018: R$ 1,425,518.53  [R$ 1,211,690.75 – R$ 1,639,346.31]
  Sep 2018: R$ 1,121,767.93  [R$   953,502.74 – R$ 1,290,033.11]
  Oct 2018: R$ 1,395,343.18  [R$ 1,186,041.70 – R$ 1,604,644.66]


## Simulación de Actualización Mensual (Incremental)
> **Sprint 2** – *Simular llegada de nuevos datos mensuales → actualizar modelo incrementalmente*

In [12]:
from src.data.monthly_agg import simulate_monthly_update

# Simular nuevo mes de datos (valores hipotéticos)
new_month = pd.DataFrame([{
    'ds':              pd.Timestamp('2018-09-01'),
    'monthly_revenue': 1_450_000.0,    # valor real que llegaría
    'monthly_orders':  5800.0,
    'monthly_customers': 5200.0,
    'monthly_items':   7200.0,
    'avg_ticket':      155.0,
    'median_ticket':   130.0,
    'avg_review_score': 4.1,
    'avg_delivery_days': 11.5,
    'avg_delay_days':  1.2,
    'pct_late_delivery': 0.08,
    'avg_freight':     15.0,
    'avg_price':       140.0,
    'total_freight':   87000.0,
    'pct_credit_card': 0.75,
    'pct_boleto':      0.18,
    'pct_installments': 0.45,
    'avg_installments': 2.1,
    'unique_categories': 68,
    'unique_sellers':  2800,
    'unique_states':   27,
}])
new_month['year_month'] = new_month['ds'].dt.to_period('M')

# Incorporar al histórico
monthly_updated = simulate_monthly_update(monthly_clean, new_month, cfg)
print(f' Tabla actualizada: {len(monthly_updated)} meses')
print(f'   Último mes: {monthly_updated["ds"].max().strftime("%b %Y")}')

2026-06-06 09:16:27.380 | INFO     | src.data.monthly_agg:simulate_monthly_update:185 - Simulando actualización mensual de datos…
2026-06-06 09:16:27.410 | INFO     | src.data.monthly_agg:add_time_series_features:119 - Features de series temporales añadidas: 44 cols totales
2026-06-06 09:16:27.412 | INFO     | src.data.monthly_agg:simulate_monthly_update:189 - Tabla actualizada: 21 meses


 Tabla actualizada: 21 meses
   Último mes: Sep 2018


## Métricas de Negocio Finales


In [13]:
from src.evaluation.metrics import compute_business_metrics, format_business_report

df_valid = df_master[
    (df_master['order_status'] == 'delivered') &
    (df_master['payment_value'] > 0)
]

biz_metrics = compute_business_metrics(df_valid, monthly)
print(format_business_report(biz_metrics))

2026-06-06 09:18:03.227 | INFO     | src.evaluation.metrics:compute_business_metrics:129 - Métricas de negocio calculadas



 MÉTRICAS DE NEGOCIO
  revenue_total                           : R$   19,881,945.07
  avg_monthly_revenue                     : R$      930,420.72
  median_monthly_revenue                  : R$    1,007,873.25
  max_monthly_revenue                     : R$    1,559,739.87
  min_monthly_revenue                     : R$           19.62
  avg_review_score                        : 4.08
  pct_5star                               : 57.08%
  pct_1star                               : 11.33%
  avg_delivery_days                       : 12.02
  pct_on_time                             : 92.68%
  avg_late_delay_days                     : 10.05
  total_unique_customers                  : 93,357
  repeat_customer_rate                    : 3.00%


## Guardar Artefactos del Pipeline


In [14]:
from src.utils.helpers import save_model, save_dataframe

save_model(forecaster_lgbm, 'lgbm_forecaster', cfg)
save_model(cleaning_pipeline, 'cleaning_pipeline', cfg)
save_dataframe(monthly_clean, 'monthly_features', cfg)

print(' Artefactos guardados:')
print('   data/models/lgbm_forecaster.pkl')
print('   data/models/cleaning_pipeline.pkl')
print('   data/processed/monthly_features.parquet')

2026-06-06 09:18:14.148 | INFO     | src.utils.helpers:save_model:52 - Modelo guardado: C:\RONALD\maestria\ProyectoG10-Sprint2\data\models\lgbm_forecaster.pkl
2026-06-06 09:18:14.159 | INFO     | src.utils.helpers:save_model:52 - Modelo guardado: C:\RONALD\maestria\ProyectoG10-Sprint2\data\models\cleaning_pipeline.pkl
2026-06-06 09:18:14.311 | INFO     | src.utils.helpers:save_dataframe:74 - DataFrame guardado: C:\RONALD\maestria\ProyectoG10-Sprint2\data\processed\monthly_features.parquet (20 filas)


 Artefactos guardados:
   data/models/lgbm_forecaster.pkl
   data/models/cleaning_pipeline.pkl
   data/processed/monthly_features.parquet


## Conclusiones del Sprint 2

### Pipeline Implementado
El Sprint 2 completó el pipeline de datos reproducible con los siguientes pasos:

1. **Limpieza** – Clipado de outliers (p1-p99), imputación por mediana, agrupamiento de categorías raras  
2. **Selección de variables** – Flujo secuencial: Missing → PSI → Correlación (0.95) → Univariante (MI, 0.30) → Wrapper (RFECV)  
3. **Hiperparametrización** – Optuna con TimeSeriesSplit CV, minimizando RMSE  
4. **Modelo final** – LightGBM con actualización incremental mensual  

### Métricas Alcanzadas
| Métrica | Baseline (MA-3M) | Modelo Final |
|---------|------------------|--------------|
| RMSE    | ~96,804          | Por evaluar  |
| MAPE    | 7.12%            | Objetivo < 10% |

### Próximos Pasos (Sprints 3-4)
- Sprint 3: Dashboard completo + API en producción  
- Sprint 4: Consideraciones éticas, gobernanza y sostenibilidad del modelo  
